# SimpleFunctions: Prediction Market Data for AI Agents

Real-time probabilities from 9,706 prediction markets (Kalshi + Polymarket). No API key needed for public endpoints.

- **Website**: [simplefunctions.dev](https://simplefunctions.dev)
- **Docs**: [simplefunctions.dev/docs](https://simplefunctions.dev/docs)
- **GitHub**: [spfunctions/simplefunctions-cli](https://github.com/spfunctions/simplefunctions-cli)
- **PyPI**: [simplefunctions](https://pypi.org/project/simplefunctions/)
- **npm**: [@spfunctions/cli](https://www.npmjs.com/package/@spfunctions/cli)

## 1. World State — What's happening right now?

~800 tokens of calibrated probabilities from prediction markets. Covers geopolitics, economy, energy, elections, crypto, tech.

In [ ]:
import requests

# Get the world state — no auth, no setup
world = requests.get('https://simplefunctions.dev/api/agent/world').text
print(world)

## 2. World Delta — What changed recently?

In [ ]:
# What changed in the last hour? (~30-50 tokens vs 800 for full state)
delta = requests.get('https://simplefunctions.dev/api/agent/world/delta', params={'since': '1h'}).text
print(delta)

## 3. Search Markets

In [ ]:
# Search Kalshi + Polymarket by keyword
results = requests.get('https://simplefunctions.dev/api/public/scan', params={'q': 'oil'}).json()

for m in results.get('results', [])[:10]:
    venue = m.get('venue', '?')
    title = m.get('title', m.get('question', '?'))
    price = m.get('yesPrice', m.get('yes_price', '?'))
    print(f"  [{venue}] {title}: {price}c")

## 4. SF Prediction Market Index

Composite index computed from 548 orderbook-tracked tickers.

In [ ]:
idx = requests.get('https://simplefunctions.dev/api/public/index').json()
i = idx.get('index', {})
print(f"Uncertainty:      {i.get('uncertainty', '?')}/100")
print(f"Geopolitical Risk: {i.get('geopolitical', '?')}/100")
print(f"Momentum:         {i.get('momentum', '?')}")
print(f"Activity:         {i.get('activity', '?')}/100")

## 5. Trade Ideas

S&T-style pitches with conviction, catalyst, direction, and risk.

In [ ]:
ideas = requests.get('https://simplefunctions.dev/api/public/ideas').json()
idea_list = ideas if isinstance(ideas, list) else ideas.get('ideas', ideas.get('items', []))

for idea in idea_list[:5]:
    conv = idea.get('conviction', '?').upper()
    direction = idea.get('direction', '?')
    headline = idea.get('headline', '?')
    print(f"  [{conv}] {direction} — {headline}")
    if idea.get('pitch'):
        print(f"    {idea['pitch'][:120]}...")
    print()

## 6. Cross-Market Contagion

When market A moves, which connected markets SHOULD move but haven't yet? The gap is a potential edge.

In [ ]:
contagion = requests.get('https://simplefunctions.dev/api/public/contagion', params={'window': '24h'}).json()

for trigger in contagion.get('triggers', [])[:3]:
    print(f"TRIGGER: {trigger.get('title', '?')} ({trigger.get('delta', '?')}c move)")
    for lag in trigger.get('lagging', [])[:2]:
        print(f"  → {lag.get('title', '?')} (gap: {lag.get('gap', '?')}c)")
    print()

## 7. Market Diff — Derivatives Over Time

Price, volume, spread, and depth changes between two time windows. Detects divergence signals.

In [ ]:
diff = requests.get('https://simplefunctions.dev/api/public/diff', params={'topic': 'iran', 'window': '24h'}).json()

for m in diff.get('markets', [])[:5]:
    title = m.get('title', '?')
    pd = m.get('priceDelta', 0)
    signals = m.get('signals', [])
    sig_str = ', '.join(signals) if signals else 'none'
    print(f"  {title}: {pd:+d}c | signals: {sig_str}")

## 8. Daily Briefing

LLM-synthesized topic briefing with key movers, sentiment, and outlook.

In [ ]:
briefing = requests.get('https://simplefunctions.dev/api/public/briefing', params={'topic': 'oil'}).json()

print(f"Topic: {briefing.get('topic', '?')}")
print(f"Headline: {briefing.get('headline', '?')}")
print()
print(briefing.get('summary', 'No summary available'))

## 9. Build a LangChain Agent (optional)

Wrap any endpoint as a LangChain tool. Requires `OPENAI_API_KEY`.

In [ ]:
# Uncomment to install + run
# !pip install langchain langchain-openai

# import os
# os.environ['OPENAI_API_KEY'] = 'sk-...'  # your key

# from langchain.tools import tool
# from langchain_openai import ChatOpenAI
# from langchain.agents import AgentExecutor, create_openai_tools_agent
# from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# @tool
# def world_state() -> str:
#     """Get real-time world state from prediction markets."""
#     return requests.get('https://simplefunctions.dev/api/agent/world').text

# @tool
# def scan_markets(query: str) -> str:
#     """Search prediction markets by keyword."""
#     return requests.get('https://simplefunctions.dev/api/public/scan', params={'q': query}).text

# prompt = ChatPromptTemplate.from_messages([
#     ('system', 'You are a prediction market analyst. Cite specific prices.'),
#     MessagesPlaceholder('chat_history', optional=True),
#     ('human', '{input}'),
#     MessagesPlaceholder('agent_scratchpad'),
# ])

# llm = ChatOpenAI(model='gpt-4o', temperature=0)
# agent = create_openai_tools_agent(llm, [world_state, scan_markets], prompt)
# executor = AgentExecutor(agent=agent, tools=[world_state, scan_markets], verbose=True)
# result = executor.invoke({'input': 'What is the probability of a US recession in 2026?'})
# print(result['output'])

---

## Next Steps

- **MCP**: `claude mcp add simplefunctions --url https://simplefunctions.dev/api/mcp/mcp`
- **CLI**: `npm i -g @spfunctions/cli && sf agent`
- **Python SDK**: `pip install simplefunctions`
- **Scaffold a project**: `npx create-prediction-market-agent my-agent`
- **Docs**: [simplefunctions.dev/docs](https://simplefunctions.dev/docs)
- **Awesome List**: [awesome-prediction-markets](https://github.com/spfunctions/awesome-prediction-markets)